In [2]:
pip install scipy pyarrow pyyaml

  Using cached scipy-1.17.1-cp313-cp313-win_amd64.whl.metadata (60 kB)
  Using cached pyyaml-6.0.3-cp313-cp313-win_amd64.whl.metadata (2.4 kB)
Using cached scipy-1.17.1-cp313-cp313-win_amd64.whl (36.5 MB)
Using cached pyyaml-6.0.3-cp313-cp313-win_amd64.whl (154 kB)

   ---------------------------------------- 0/2 [scipy]
   ---------------------------------------- 0/2 [scipy]
   ---------------------------------------- 0/2 [scipy]
   ---------------------------------------- 0/2 [scipy]
   ---------------------------------------- 0/2 [scipy]
   ---------------------------------------- 0/2 [scipy]
   ---------------------------------------- 0/2 [scipy]
   ---------------------------------------- 0/2 [scipy]
   ---------------------------------------- 0/2 [scipy]
   ---------------------------------------- 0/2 [scipy]
   ---------------------------------------- 0/2 [scipy]
   ---------------------------------------- 0/2 [scipy]
   ---------------------------------------- 0/2 [scipy]
   --


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import sys
sys.path.append('../')

from backend.ml.adapters.dataset_adapter import DatasetAdapter
from collections import Counter
import numpy as np

In [8]:
import shutil
from pathlib import Path

cache_file = Path("data/cache/86643cb3c115.pkl")
if cache_file.exists():
    cache_file.unlink()
    print("Cache supprimé")
else:
    print("Fichier non trouvé")

Cache supprimé


In [9]:
adapter_vbl = DatasetAdapter('../configs/datasets/vbl_va001.yaml')

# max_files=20 → on ne charge que 20 fichiers par classe
# parfait pour développement sans télécharger les 4 Go
signals_vbl = adapter_vbl.load(
    '../data/raw/VBL-VA001.zip/',
    max_files=20,
    use_cache=True
)

print(f"\nNombre de signaux     : {len(signals_vbl)}")
print(f"Shape fenêtres        : {signals_vbl[0].signals.shape}")
print(f"  → (n_fenêtres, {signals_vbl[0].signals.shape[1]} points, {signals_vbl[0].signals.shape[2]} canaux)")
print(f"Fréquence cible       : {signals_vbl[0].sampling_rate} Hz")
print(f"Label                 : {signals_vbl[0].label}")

print("\nDistribution des classes :")
for label, count in Counter(s.label for s in signals_vbl).items():
    print(f"  {label:20s} : {count}")

  Classes trouvées dans le ZIP : ['bearing', 'misalignment', 'normal', 'unbalance']
  Stats normalisation calculées sur 50 fichiers 'normal'
  [bearing             ] 20 fichiers...
  [misalignment        ] 20 fichiers...
  [normal              ] 20 fichiers...
  [unbalance           ] 20 fichiers...

  Total : 80 signaux chargés depuis ZIP
  Sauvegarde cache → 86643cb3c115.pkl

Nombre de signaux     : 80
Shape fenêtres        : (123, 1024, 3)
  → (n_fenêtres, 1024 points, 3 canaux)
Fréquence cible       : 12800 Hz
Label                 : bearing

Distribution des classes :
  bearing              : 20
  misalignment         : 20
  normal               : 20
  unbalance            : 20


In [10]:
# Un fichier VBL original = 20 000 Hz × 5 sec = 100 000 points
# Après resampling à 12 800 Hz = 12 800 × 5 = 64 000 points
# Après fenêtrage (1024 pts, hop 512) = ~124 fenêtres

sig = signals_vbl[0]
n_windows   = sig.signals.shape[0]
window_size = sig.signals.shape[1]
n_channels  = sig.signals.shape[2]
duree_sec   = (n_windows * 512 + window_size) / 12_800

print(f"Fenêtres par signal   : {n_windows}")
print(f"Points par fenêtre    : {window_size}")
print(f"Canaux (axes x,y,z)   : {n_channels}")
print(f"Durée reconstituée    : {duree_sec:.2f} sec (doit être ≈ 5 sec)")

# Vérification normalisation — moyenne ≈ 0, std ≈ 1
flat = sig.signals.reshape(-1, n_channels)
print(f"\nMoyenne par canal     : {flat.mean(axis=0).round(3)}")
print(f"Std par canal         : {flat.std(axis=0).round(3)}")

Fenêtres par signal   : 123
Points par fenêtre    : 1024
Canaux (axes x,y,z)   : 3
Durée reconstituée    : 5.00 sec (doit être ≈ 5 sec)

Moyenne par canal     : [-0.01   0.012  0.004]
Std par canal         : [0.986 2.389 3.685]


In [ ]:
adapter_cmapss = DatasetAdapter('../configs/datasets/cmapss.yaml')
signals_cmapss = adapter_cmapss.load(
    '../data/raw/cmapss/',
    use_cache=True
)

print(f"\nNombre de fenêtres    : {len(signals_cmapss)}")
print(f"Shape d'une fenêtre   : {signals_cmapss[0].signals.shape}")
print(f"  → (1, 30 cycles, 21 capteurs)")
print(f"RUL exemple           : {signals_cmapss[0].rul:.1f} cycles")
print(f"Label synthétique     : {signals_cmapss[0].label}")

print("\nDistribution des classes CMAPSS :")
for label, count in Counter(s.label for s in signals_cmapss).items():
    print(f"  {label:25s} : {count}")

  Chargement CMAPSS : train_FD001.txt...
  Total : 17731 fenêtres CMAPSS chargées
  Sauvegarde cache → 2ffb1e19adf2.pkl

Nombre de fenêtres    : 17731
Shape d'une fenêtre   : (1, 30, 21)
  → (1, 30 cycles, 21 capteurs)
RUL exemple           : 125.0 cycles
Label synthétique     : sain

Distribution des classes CMAPSS :
  sain                      : 9631
  degradation_precoce       : 4000
  degradation_avancee       : 2500
  critique                  : 1600


In [11]:
# Les deux doivent avoir des signaux normalisés (moyenne ≈ 0)
# et des shapes cohérentes pour le FeatureExtractor

vbl_shape    = signals_vbl[0].signals.shape
cmapss_shape = signals_cmapss[0].signals.shape

print("=== Validation de cohérence ===")
print(f"VBL shape    : {vbl_shape}")
print(f"CMAPSS shape : {cmapss_shape}")
print()
print("Note : les deux shapes sont différentes (normal)")
print("→ VBL    = fenêtres sur signal temporel haute fréquence")
print("→ CMAPSS = fenêtres sur cycles consécutifs (pas de Hz)")
print()
print("Le FeatureExtractor devra traiter chaque type séparément.")
print("✓ Architecture validée — prêt pour l'étape suivante.")

=== Validation de cohérence ===
VBL shape    : (123, 1024, 3)
CMAPSS shape : (1, 30, 21)

Note : les deux shapes sont différentes (normal)
→ VBL    = fenêtres sur signal temporel haute fréquence
→ CMAPSS = fenêtres sur cycles consécutifs (pas de Hz)

Le FeatureExtractor devra traiter chaque type séparément.
✓ Architecture validée — prêt pour l'étape suivante.
